# Final results (after kfold hyperparam. selection)

### Load results from wandb

In [3]:
%load_ext autoreload
%autoreload 2

import numpy as np
from utils_table_generator import *
from utils_final_results import *

wandb_username = "lcornelis"  # Change this to your W&B username if needed
wandb_project = "ProteoFinalAll"  # Change this to your W&B project name if needed
metric = "mse"  # Change this to the metric you want to extract (e.g., "mae", "mse", etc.)
original_units = True  # Set to True if you want to convert back to original units
csv_filename = "final_results.csv"  # Output CSV filename
save_csv = False  # Set to True if you want to save the grouped results to a CSV file

df = load_results_dataframe(wandb_username, wandb_project, original_units=original_units, metric=metric, csv_filename=csv_filename, save_csv=save_csv)
# df, grouped, best_configs, summary, table = generate_table(df, save_csv=save_csv, csv_filename=csv_filename)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
▶ Number of runs fetched from W&B: 5
▶ After building df, df.shape = (3, 168)


### Filter dataframe to see final performances and checkpoints

In [3]:
columns_to_keep = ["dataset", "model", "test_mae", "checkpoint"]
filtered_df = df[columns_to_keep]
pd.set_option('display.max_colwidth', None)
filtered_df

,dataset,model,test_mae,checkpoint
0,pointcloud,mlp,11.898328,/scratch/lcornelis/outputs/checkpoints/epoch_060-v2.ckpt
1,wgcna,gat,13.948260,/scratch/lcornelis/outputs/checkpoints/epoch_003-v1.ckpt
2,wgcna,gcn,13.531629,/scratch/lcornelis/outputs/checkpoints/epoch_071.ckpt


### Load the corresponding config file and checkpoint

Here we assume a single run per (model, adj_metric) pair

In [1]:
from utils_final_results import get_config_and_checkpoint, load_model_checkpoint, load_dataset

model = "gcn"
adj_metric = "wgcna"
checkpoint = torch.load("/scratch/lcornelis/outputs/checkpoints/epoch_060-v2.ckpt")
print(type(torch.load("/scratch/lcornelis/outputs/checkpoints/epoch_060-v2.ckpt")))
print(checkpoint["hyper_parameters"].keys())
print(OmegaConf.create(checkpoint["hyper_parameters"]["cfg"]))


cfg, checkpoint = get_config_and_checkpoint(model, adj_metric, df)
model = load_model_checkpoint(cfg, checkpoint)


/home/lcornelis/anaconda3/envs/proteo/lib/python3.11/site-packages/torch_geometric/typing.py:155: UserWarning: An issue occurred while importing 'torch-spline-conv'. Disabling its usage. Stacktrace: /home/lcornelis/anaconda3/envs/proteo/lib/python3.11/site-packages/torch_spline_conv/_version_cuda.so: undefined symbol: _ZN3c1017RegisterOperatorsD1Ev
  warnings.warn(
/home/lcornelis/code/TopoProteo/tutorials/utils_final_results.py:55: UserWarning: 
The version_base parameter is not specified.
Please specify a compatability version level, or None.
Will assume defaults for version 1.1
  initialize(config_path="../configs", job_name="job")


NameError: name 'torch' is not defined

### Load dataset

In [4]:
adj_metric = "spearman_correlation"
adj_threshold = 0.5
kfold = True
num_folds = 5
fold = 0

train_dataset, val_dataset, _ = load_dataset(adj_metric, adj_threshold, kfold=kfold, num_folds=num_folds, fold=fold)

Processed file names: ['FTD_y_val_nfl_spearman_correlation_adj_thresh_0.5_num_nodes_7258_mutation_GRN,MAPT,C9orf72,CTL_csf_sex_M,F_random_state_42_5fold_0_train.pt', 'FTD_y_val_nfl_spearman_correlation_adj_thresh_0.5_num_nodes_7258_mutation_GRN,MAPT,C9orf72,CTL_csf_sex_M,F_random_state_42_5fold_0_val.pt']
Loading data from: /scratch/lcornelis/data/data_louisa/FTD/processed/FTD_y_val_nfl_spearman_correlation_adj_thresh_0.5_num_nodes_7258_mutation_GRN,MAPT,C9orf72,CTL_csf_sex_M,F_random_state_42_5fold_0_train.pt
Processed file names: ['FTD_y_val_nfl_spearman_correlation_adj_thresh_0.5_num_nodes_7258_mutation_GRN,MAPT,C9orf72,CTL_csf_sex_M,F_random_state_42_5fold_0_train.pt', 'FTD_y_val_nfl_spearman_correlation_adj_thresh_0.5_num_nodes_7258_mutation_GRN,MAPT,C9orf72,CTL_csf_sex_M,F_random_state_42_5fold_0_val.pt']
Loading data from: /scratch/lcornelis/data/data_louisa/FTD/processed/FTD_y_val_nfl_spearman_correlation_adj_thresh_0.5_num_nodes_7258_mutation_GRN,MAPT,C9orf72,CTL_csf_sex_M,F_r

In [ ]:
train_dataset

FTDDataset(143)

In [6]:
val_dataset

FTDDataset(36)